# Hurricane Tracker -- Interactive
This demonstrates the use of input and display widgets in a Jupyter Notebook

In [ ]:
# Import packages
from pathlib import Path
import pandas as pd
import seaborn as sns
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

# GIS pakcages
from arcgis import GIS
from arcgis.features import FeatureLayer
from arcgis.geometry import Polyline

#Set paths
raw_folder_path = Path.cwd().parent / 'data' / 'raw'
processed_folder_path = Path.cwd().parent / 'data' / 'processed'

In [ ]:
#Set model layers
ibtracs_NA_points = str(raw_folder_path /'IBTrACS_NA.shp')
usa_counties = 'https://services.arcgis.com/P3ePLMYs2RVChkJx/arcgis/rest/services/USA_Counties_Generalized_Boundaries/FeatureServer/0'
storm_track_points =  "memory\\track_points"
storm_track_line = "memory\\Tracklines"

In [ ]:
#Convert storm names to spatial dataframe
sdf_allstorms = pd.DataFrame.spatial.from_featureclass(ibtracs_NA_points)

In [ ]:
#Get list of seasons
all_seasons = sdf_allstorms['SEASON'].unique()

In [ ]:
#Function to extract points for a given season
def get_stormspoints(the_season=2000):
    sdf_select_points = sdf_allstorms.loc[sdf_allstorms['SEASON']==the_season]
    print(f"{sdf_select_points.shape[0]} storm points in {the_season}")
    return sdf_select_points

In [ ]:
#Function to extract counties for a given season and name
the_season = 2005
the_name = 'KATRINA'
#Select the points
sdf_storm_points = sdf_allstorms.loc[
    (sdf_allstorms['SEASON']==the_season) &
    (sdf_allstorms['NAME']==the_name)
    ]
#Sort points on ISO_TIME
sdf_storm_points = sdf_storm_points.sort_values(by='ISO_TIME')
sdf_storm_points.head(2)

In [ ]:
#Convert the point coordinates to a line geometry
coords = sdf_storm_points['SHAPE']


In [ ]:
# Create a dictionary of storm names for each storm season

storm_dict = {}
for storm_season in range(2000,2004):
    storm_names = []
    with arcpy.da.SearchCursor(
        in_table = ibtracs_NA_points,
        field_names = ['NAME'], 
        where_clause=f"SEASON = {storm_season}") as cursor:
        for row in cursor:
            if row[0] not in storm_names and row[0] != 'UNNAMED':
                storm_names.append(row[0])
    storm_dict[storm_season] = storm_names